# hindustani-raag-small **v1**

Same source audios as v0 plus six newly downloaded ones, rechunked. Three changes:

1. **skip list** — the 20 videos hand-flagged as unusable in `annotate_tonics.py` are dropped.
2. **more / longer chunks** — 5 train and 3 test chunks per video, each between 20 s and 60 s
   (v0 had 3 / 2 chunks of `0.01 * duration`, which for a 10-minute clip was only 6 s).
3. **tonic as a feature** — every row carries the hand-annotated Sa of its source recording
   (`tonics.csv`). This is *input*, not a label: a model may use it at inference time too.

The dataset has exactly three columns — `audio`, `label`, `tonic_hz` — so v0 consumers see one
added column and nothing removed. The train/test designation is carried over from v0 unchanged;
videos v0 never had are designated by `annotate_tonics_new.py`.

This pushes a **new revision of the same repo**, `neerajaabhyankar/hindustani-raag-small`. Old
revisions stay resolvable, so anything pinning `DATASET_REVISION` is unaffected.

In [ ]:
import os
import csv
import regex as re
import numpy as np
import soundfile as sf
from typing import Dict, List, Tuple

## Config

In [ ]:
DOWNLOAD_DIR = "/Users/neerajaabhyankar/Repos/icm-shruti-analysis/raag-identifier/hindustani-raag-fullaudios"
DATASET_DIR_V0 = "/Users/neerajaabhyankar/Repos/icm-shruti-analysis/raag-identifier/hindustani-raag-small"
DATASET_DIR = "/Users/neerajaabhyankar/Repos/icm-shruti-analysis/raag-identifier/hindustani-raag-small-v1"
TONICS_CSV = "/Users/neerajaabhyankar/Repos/icm-shruti-analysis/raag-identifier/raagdataset/tonics.csv"

HUB_REPO_ID = "neerajaabhyankar/hindustani-raag-small"

os.makedirs(DATASET_DIR, exist_ok=True)

## Utils

In [ ]:
URL_PATTERN = re.compile(r"\[([^\]]+)\]")
CHUNK_PATTERN = re.compile(r"^(train|test)_\[(.+)\]_chunk(\d+)\.mp3$")


def video_id(filename: str) -> str:
    """The youtube id is the last [...] group in the downloaded filename."""
    return re.findall(URL_PATTERN, filename)[-1]


def read_audio_section(filename, start_time, stop_time):
    """Read [start_time, stop_time) seconds. Frame-accurate, unlike v0's int() version."""
    track = sf.SoundFile(filename)
    if not track.seekable():
        raise ValueError("Not compatible with seeking")

    sr = track.samplerate
    start_frame = int(round(sr * start_time))
    frames_to_read = int(round(sr * (stop_time - start_time)))
    track.seek(start_frame)
    audio_section = track.read(frames_to_read)

    return audio_section, sr


def audio_duration(path) -> float:
    f = sf.SoundFile(path)
    return f.frames / f.samplerate


def listdir(path) -> List[str]:
    """sorted, no dotfiles (.DS_Store would otherwise shift indices around)."""
    return sorted(n for n in os.listdir(path) if not n.startswith("."))

## 1. Skip list

Copied verbatim from `annotate_tonics.py :: load_skip_list()` — videos that turned out to be
speech, a wrong raag, or otherwise unusable while hand-annotating tonics.

In [ ]:
SKIP_LIST = [
    "0GkzyJbxCMA",
    "5z_sUCM__uc",
    "ZWQDBjkAW-w",
    "PKUO-nqbABg",
    "A3SneqtmNog",
    "azznhT3coJE",
    "FZtu7xweL0Y",
    "M3DE88z0Nv8",
    "MN7VkVPaytM",
    "85y7SzjvCjk",
    "aUWTwQk_kUY",
    "e3iMnt28DLM",
    "k86EmhqcpUA",
    "nI7v5zCLawk",
    "2gJbTYxmqMA",
    "LknyChMkj3g",
    "iOtSjWMKdZI",
    "AVxwGAy1ygM",
    "G_wmgnxgK0w",
    "egHCxISQG9o",
    "SuNEIYlw9z0",
]
SKIP = set(SKIP_LIST)
len(SKIP)

## 2. Tonics

`tonics.csv` is keyed by video, not by clip — chunks of one recording share a tonic. A video
with no row here is simply not in the dataset: the tonic is a required column, so an
un-annotated video has nothing to contribute yet. Section 3 lists any such videos.

In [ ]:
def load_tonics(path=TONICS_CSV) -> Dict[str, dict]:
    with open(path) as fh:
        return {r["video"]: r for r in csv.DictReader(fh) if r.get("video")}


TONICS = load_tonics()
print(f"{len(TONICS)} annotated videos")
next(iter(TONICS.items()))

## 3. Train / test split

Two sources, in this order of authority:

1. **the v0 dataset folder** — `test_[<video>]_chunk*.mp3` names v0's test video. Authoritative
   for every video v0 already had, so v1 and v0 numbers stay comparable.
2. **`tonics.csv`'s `clip` column** — for videos v0 never had. `annotate_tonics_new.py` records
   the chunk it played, e.g. `test_[<video>]_chunk1.mp3`, and that prefix is the designation.
   This is what keeps newly downloaded audio from silently defaulting to train.

v0's folder wins for videos it knows, for two reasons. Its `clip` prefixes in `tonics.csv` are
a snapshot of an *earlier* v0 split — three of them (`9JLAuhZscAE` Bairagi, `9GlMrtTfSD0`
Bhairav, `nieORT40gzc` Marwa) disagree with the v0 folder as it stands today — and `tonics.csv`
is keyed by video, so for a video filed under two raags it can only record one designation.

In [ ]:
def split_from_v0(v0_dir=DATASET_DIR_V0) -> Dict[Tuple[str, str], str]:
    """{(raag, video): "train"|"test"} as designated in v0.

    Keyed by *both*, not by video alone: seven videos sit in two raag folders each, and v0 can
    designate the two copies differently (9JLAuhZscAE is test under Bairagi and train under
    Bhairav). Keying by video alone silently lets one raag's designation overwrite the other's.
    """
    out = {}
    for raag in listdir(v0_dir):
        if not os.path.isdir(os.path.join(v0_dir, raag)):
            continue
        for fname in listdir(os.path.join(v0_dir, raag)):
            m = CHUNK_PATTERN.match(fname)
            if m:
                out[(raag, m.group(2))] = m.group(1)
    return out


def split_from_tonics(tonics=None) -> Dict[str, str]:
    """{video: "train"|"test"} from the clip an annotation played."""
    tonics = TONICS if tonics is None else tonics
    return {
        v: ("test" if r.get("clip", "").startswith("test_") else "train")
        for v, r in tonics.items()
    }


V0_SPLIT = split_from_v0()
V0_VIDEOS = {video for _raag, video in V0_SPLIT}
TONIC_SPLIT = split_from_tonics()
print(f"v0 designates {len(V0_SPLIT)} (raag, video) pairs over {len(V0_VIDEOS)} videos; "
      f"tonics.csv designates {len(TONIC_SPLIT)}")
print(f"new since v0, designated by tonics.csv: {sorted(set(TONIC_SPLIT) - V0_VIDEOS)}")

In [ ]:
def build_split(download_dir=DOWNLOAD_DIR) -> Dict[str, Dict[str, List[str]]]:
    """{raag: {"train": [video...], "test": [video...], "files": {video: filename}}}"""
    split, pending = {}, []
    for raag in listdir(download_dir):
        raag_dir = os.path.join(download_dir, raag)
        if not os.path.isdir(raag_dir):
            continue

        present = {video_id(f): f for f in listdir(raag_dir) if f.lower().endswith(".mp3")}
        kept, dropped = {}, []
        for v, f in present.items():
            if v in SKIP:
                dropped.append(v)
            elif v not in TONICS:
                pending.append((raag, v))  # downloaded but not yet annotated
            else:
                kept[v] = f

        train, test = [], []
        for v in sorted(kept):
            (test if V0_SPLIT.get((raag, v), TONIC_SPLIT[v]) == "test" else train).append(v)

        split[raag] = {"train": train, "test": test, "files": kept}
        flag = "   <-- NO TEST VIDEO" if not test else ""
        print(f"{raag:18s} train {len(train):2d}  test {len(test):2d}"
              f"  (skip-listed {len(dropped)}){flag}")

    if pending:
        print(f"\n{len(pending)} downloaded video(s) have no tonic yet and are NOT in v1 — "
              f"run annotate_tonics_new.py:")
        for raag, v in pending:
            print(f"    {raag:18s} {v}")
    return split


SPLIT = build_split()

## 4. Chunk plan

Relative start points in the recording, and a length rule.

v0's `0.01 * duration` gave chunks far too short to hear a raag in (6 s for a 10-minute clip),
so the length is now `0.01 * duration` **clamped to [20 s, 60 s]** — long recordings get up to
a minute, short ones still get a usable 20 s. A chunk that would run past the end is slid back
to end exactly at the end; recordings shorter than 20 s are skipped entirely.

Start points avoid the first 20% (alap/announcements/applause) and the last 10%.

`annotate_tonics_new.py` mirrors these constants — if you change them here, change them there,
or a new video's tonic gets snapped against audio the dataset does not contain.

In [ ]:
TRAIN_STARTS = [0.21, 0.41, 0.56, 0.71, 0.86]
TEST_STARTS = [0.36, 0.51, 0.76]

MIN_CHUNK_SEC = 20.0
MAX_CHUNK_SEC = 60.0
CHUNK_FRACTION = 0.01


def chunk_length(duration: float) -> float:
    return float(np.clip(CHUNK_FRACTION * duration, MIN_CHUNK_SEC, MAX_CHUNK_SEC))


def chunk_bounds(duration: float, starts: List[float]) -> List[Tuple[float, float]]:
    """(start_sec, stop_sec) per relative start point, clipped to fit inside the recording."""
    length = chunk_length(duration)
    if duration < MIN_CHUNK_SEC:
        return []
    bounds = []
    for frac in starts:
        start = frac * duration
        start = min(start, duration - length)  # slide back rather than truncate
        start = max(start, 0.0)
        bounds.append((start, start + length))
    return bounds


# sanity: what the rule does at a few durations
for d in [15, 60, 600, 1800, 3600, 7200]:
    print(f"{d:5d}s recording -> {chunk_length(d):5.1f}s chunks, "
          f"train bounds {[(round(a), round(b)) for a, b in chunk_bounds(d, TRAIN_STARTS)]}")

## 5. Write the chunks

Filenames follow v0 (`{split}_[{video}]_chunk{i}.mp3`) so anything that parsed v0 still parses
v1. Already-written chunks are left alone, so this cell is safe to re-run after adding audio.

In [ ]:
rows = []
n_written, n_existing, n_short = 0, 0, 0

for raag in sorted(SPLIT):
    files = SPLIT[raag]["files"]
    os.makedirs(os.path.join(DATASET_DIR, raag), exist_ok=True)

    for split_name, starts in (("train", TRAIN_STARTS), ("test", TEST_STARTS)):
        for video in SPLIT[raag][split_name]:
            src = os.path.join(DOWNLOAD_DIR, raag, files[video])
            duration = audio_duration(src)
            bounds = chunk_bounds(duration, starts)
            if not bounds:
                print(f"  too short ({duration:.1f}s), skipping: {raag}/{files[video]}")
                n_short += 1
                continue

            for ci, (start, stop) in enumerate(bounds):
                fname = f"{split_name}_[{video}]_chunk{ci}.mp3"
                writepath = os.path.join(DATASET_DIR, raag, fname)

                if os.path.exists(writepath):
                    n_existing += 1
                else:
                    audio_chunk, sr = read_audio_section(src, start, stop)
                    sf.write(writepath, audio_chunk, sr)
                    n_written += 1

                rows.append({
                    "file_name": f"{raag}/{fname}",
                    "raag": raag,
                    "split": split_name,
                    "video": video,
                    "tonic_hz": float(TONICS[video]["tonic_hz"]),
                })

    print(f"{raag:18s} {sum(r['raag'] == raag for r in rows):3d} chunks")

print(f"\n{n_written} written, {n_existing} already existed, {n_short} recordings too short")
print(f"{len(rows)} rows over {len(set(r['video'] for r in rows))} videos")

## 6. `metadata.csv`

Three columns, and no more: `file_name`, `label` (the raag, by name — readable in the viewer
and in the file itself), and `tonic_hz`. `raag`/`split`/`video` stay in `rows` in memory because
sections 7-8 need them, but they are not part of the dataset.

This file is what makes the raw folder layout self-describing on the hub. The dataset the hub
actually serves is built in section 8, where `label` becomes a `ClassLabel` — the same int
encoding, with the same names, that v0 had.

In [ ]:
METADATA_FIELDS = ["file_name", "label", "tonic_hz"]

metadata_path = os.path.join(DATASET_DIR, "metadata.csv")
with open(metadata_path, "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=METADATA_FIELDS)
    w.writeheader()
    for r in sorted(rows, key=lambda r: r["file_name"]):
        w.writerow({"file_name": r["file_name"], "label": r["raag"], "tonic_hz": r["tonic_hz"]})

print(f"wrote {len(rows)} rows -> {metadata_path}")
print("\n".join(open(metadata_path).read().split("\n")[:3]))

## 7. Sanity checks

In [ ]:
# per-raag counts
print(f"{'raag':20s} {'train':>6s} {'test':>6s} {'videos':>7s} {'tonic Hz range':>22s}")
for raag in sorted(SPLIT):
    rr = [r for r in rows if r["raag"] == raag]
    tr = sum(r["split"] == "train" for r in rr)
    te = sum(r["split"] == "test" for r in rr)
    hz = [r["tonic_hz"] for r in rr]
    rng = f"{min(hz):.1f} - {max(hz):.1f}" if hz else "MISSING"
    flag = "  <-- no test" if te == 0 else ""
    print(f"{raag:20s} {tr:6d} {te:6d} {len(set(r['video'] for r in rr)):7d} {rng:>22s}{flag}")

print(f"\ntotal: {sum(r['split'] == 'train' for r in rows)} train, "
      f"{sum(r['split'] == 'test' for r in rows)} test chunks")

In [ ]:
# a handful of videos sit in two raag folders — inherited from v0, not introduced here.
# Same audio, two labels; and if the two copies land on opposite sides of the split it is
# train/test leakage. Reported, not fixed: fixing it means deciding which raag is right.
from collections import defaultdict

by_video = defaultdict(set)
for r in rows:
    by_video[r["video"]].add((r["raag"], r["split"]))

dups = {v: sorted(s) for v, s in by_video.items() if len({raag for raag, _ in s}) > 1}
print(f"{len(dups)} video(s) appear under more than one raag:")
for v, s in dups.items():
    leak = "  <-- LEAKAGE: same audio in train and test" if len({sp for _, sp in s}) > 1 else ""
    print(f"  {v}  {s}{leak}")

In [ ]:
# no video may appear on both sides of the split
train_v = {r["video"] for r in rows if r["split"] == "train"}
test_v = {r["video"] for r in rows if r["split"] == "test"}
print(f"videos on both sides: {sorted(train_v & test_v)}")

# nothing from the skip list survived
assert not (SKIP & (train_v | test_v)), sorted(SKIP & (train_v | test_v))

# every raag has test data
no_test = [raag for raag in SPLIT if not any(r["raag"] == raag and r["split"] == "test" for r in rows)]
assert not no_test, f"raags with no test chunks: {no_test}"

# every row has a tonic, and it is in a plausible band for a sung/played Sa
assert all(r["tonic_hz"] > 0 for r in rows)
hz = [r["tonic_hz"] for r in rows]
print(f"tonic: {min(hz):.1f} - {max(hz):.1f} Hz")

# the split is v0's, except for videos v0 never had
moved = sorted({(r["raag"], r["video"]) for r in rows
                if (r["raag"], r["video"]) in V0_SPLIT
                and V0_SPLIT[(r["raag"], r["video"])] != r["split"]})
assert not moved, f"(raag, video) pairs that changed side vs v0: {moved}"

fresh = sorted({r["video"] for r in rows if r["video"] not in V0_VIDEOS})
print(f"{len(fresh)} videos are new since v0: {fresh}")

print("\nall checks passed")

In [ ]:
# spot-check by ear before pushing
import IPython.display as ipd

sample = rows[len(rows) // 2]
print(sample)
ipd.Audio(os.path.join(DATASET_DIR, sample["file_name"]))

## 8. Build the `DatasetDict`

Built straight from `rows` rather than from `audiofolder`, so the schema is exactly what was
intended and nothing is inferred:

* `audio` — as v0.
* `label` — `ClassLabel` over the sorted raag names, so `ds["train"][i]["label"]` is the same
  int it was in v0 and `features["label"].names` still resolves it to the raag. The viewer
  shows the name.
* `tonic_hz` — the one new column, `float32`.

Also note v1 ships real `train` / `test` splits; v0 put everything in `train` and left the
split encoded in the filename prefix.

In [ ]:
from datasets import Dataset, DatasetDict, Audio, ClassLabel, Value, Features

RAAG_NAMES = sorted(SPLIT)  # label index == position here, same order v0's audiofolder used

FEATURES = Features({
    "audio": Audio(),
    "label": ClassLabel(names=RAAG_NAMES),
    "tonic_hz": Value("float32"),
})


def to_dataset(split_name):
    rr = sorted((r for r in rows if r["split"] == split_name), key=lambda r: r["file_name"])
    return Dataset.from_dict(
        {
            "audio": [os.path.join(DATASET_DIR, r["file_name"]) for r in rr],
            "label": [RAAG_NAMES.index(r["raag"]) for r in rr],
            "tonic_hz": [r["tonic_hz"] for r in rr],
        },
        features=FEATURES,
    )


ds = DatasetDict({name: to_dataset(name) for name in ("train", "test")})
print(ds)

row = ds["test"][0]
print("label:", row["label"], "->", ds["test"].features["label"].names[row["label"]],
      "| tonic_hz:", row["tonic_hz"], "| sr:", row["audio"]["sampling_rate"])

## 9. Push to hub

Two commits to `neerajaabhyankar/hindustani-raag-small`:

1. **`push_to_hub`** writes the parquet under `data/` plus the dataset card YAML. This is what
   `load_dataset(HUB_REPO_ID)` and the viewer read.
2. **`upload_folder`** puts the raw `<Raag>/*.mp3` + `metadata.csv` back at the repo root, the
   same layout v0 had, with **`delete_patterns`** so v0's chunks that v1 no longer contains
   (skip-listed videos, and v0's shorter chunk numbering) are removed in the same commit
   instead of lingering as orphans. Without it the repo would keep serving v0 audio files
   alongside v1's.

Old revisions are untouched, so every `DATASET_REVISION` pin in the repo keeps resolving to v0.
Print the new sha if you want to pin v1.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HUB_REPO_ID, repo_type="dataset", exist_ok=True)

In [ ]:
commit = ds.push_to_hub(
    repo_id=HUB_REPO_ID,
    commit_message="v1: tonic_hz column + train/test splits + data fixes",
)
print(commit)

In [ ]:
# raw layout at the repo root, exactly where v0 had it; delete_patterns prunes v0 leftovers
api.upload_folder(
    folder_path=DATASET_DIR,
    repo_id=HUB_REPO_ID,
    repo_type="dataset",
    allow_patterns=["*/*.mp3", "metadata.csv"],
    delete_patterns=["*/*.mp3", "metadata.csv"],
    commit_message="v1: raw chunks + metadata.csv (v0 chunks removed)",
)

In [ ]:
# final check: round-trip from the hub
from datasets import load_dataset

check = load_dataset(HUB_REPO_ID)
print(check)
print({k: v for k, v in check["test"][0].items() if k != "audio"})
print(check["test"].features)

## 10. Dataset card

`push_to_hub` writes the YAML header (splits, features, sizes); this replaces the *body* below
it, leaving that header intact — so the viewer keeps working. The body is where the
backward-compatibility notes go, since anyone pulling the update reads the card first.

In [ ]:
CARD_BODY = f"""# hindustani-raag-small

{len(RAAG_NAMES)} raags of Hindustani classical music, chunked from YouTube recordings, each
chunk carrying the hand-annotated tonic (Sa) of the recording it came from.

| | |
|---|---|
| raags | {len(RAAG_NAMES)} |
| train clips | {len(ds["train"])} |
| test clips | {len(ds["test"])} |
| clip length | {MIN_CHUNK_SEC:.0f}-{MAX_CHUNK_SEC:.0f} s |
| source videos | {len(set(r["video"] for r in rows))} |

## Columns

* `audio` — an mp3 chunk, {MIN_CHUNK_SEC:.0f}-{MAX_CHUNK_SEC:.0f} s, cut from a full recording.
* `label` — `ClassLabel` over the {len(RAAG_NAMES)} raag names.
* `tonic_hz` — the recording's Sa in Hz, annotated by ear and snapped to a peak of the
  recording's own pitch histogram. **This is an input feature, not a label**: it describes the
  performance's tuning, and a model may legitimately use it at inference time. Chunks from one
  recording share a tonic.

The repo also keeps the raw layout at the root — `<Raag>/{{train,test}}_[<video-id>]_chunk<n>.mp3`
plus `metadata.csv` — for anyone who would rather stream the files than the parquet.

## v1 — what changed, and what breaks

**Unchanged:** the {len(RAAG_NAMES)} raag names and their `label` integer encoding; the
`<Raag>/` folder layout of the raw files; the `{{split}}_[{{video}}]_chunk{{n}}.mp3` naming;
which videos are train and which are test.

**Breaking, for code that loads the dataset unpinned:**

1. **Splits.** v0 was one flat `train` split with test clips distinguished only by the
   `test_` filename prefix. v1 has real `train` and `test` splits, so `ds["train"]` is now
   train-only and **every row index has moved**. Anything that cached per-row artifacts keyed
   by index (embeddings named `train_<i>.npz`, for instance) will silently mis-pair them with
   labels. Recompute, do not reuse.
2. **The audio itself is different.** Chunks are {MIN_CHUNK_SEC:.0f}-{MAX_CHUNK_SEC:.0f} s
   taken at new offsets, where v0's were `0.01 x duration` (~6 s for a 10-minute recording),
   and there are 5 train / 3 test chunks per video where v0 had 3 / 2. All cached features
   are invalid regardless of indexing.
3. **20 videos were dropped** — flagged during tonic annotation as speech, the wrong raag, or
   otherwise unusable — and 6 were added, to give six raags a test recording they lacked.
4. **`tonic_hz` is new.** Purely additive.

**Nothing pinned breaks.** v0 remains at its own revision; `load_dataset(..., revision=...)`
against the old sha keeps returning v0 exactly.
"""

print(CARD_BODY[:600], "...")

In [ ]:
from huggingface_hub import DatasetCard

card = DatasetCard.load(HUB_REPO_ID, repo_type="dataset")  # keeps the generated YAML header
card.text = CARD_BODY
card.push_to_hub(HUB_REPO_ID, repo_type="dataset")
print(f"card updated -> https://huggingface.co/datasets/{HUB_REPO_ID}")